# LFW 매니페스트 생성

LFW deep-funneled 이미지를 전부 확인한 뒤 identity 단위로 `development / calibration / test`를 분리하고, 다음 세 입력을 생성합니다.

- `face_manifest.csv`: `image_id, identity_id, split, image_path`
- `gallery_identities.txt`: test split에서 등록 인물로 사용할 ID
- `unknown_unknown_identities.txt`: test split의 unknown unknown ID

`WRITE_OUTPUTS=False`여도 전체 이미지 탐색, 경로·중복 검사, identity 누수 검사, gallery 후보 수 검사를 모두 수행합니다. 파일만 저장하지 않습니다. 검사가 통과한 뒤 `True`로 바꾸어 다시 위에서 아래로 실행하십시오.

## 중단 후 재시작

- 매니페스트 생성 셀 이전/도중에 중단: 커널을 재시작하고 첫 코드 셀부터 다시 실행합니다.
- 저장 셀에서 중단: 출력 폴더를 확인한 뒤 `OVERWRITE=True`로 바꾸고 첫 코드 셀부터 다시 실행합니다. 각 파일은 임시 파일 작성 후 교체됩니다.
- 다른 seed나 분할 비율로 바꿀 때는 기존 결과를 섞지 말고 별도 `OUTPUT_DIR`을 사용하십시오.


In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. D:/ronbun 안에서 노트북을 실행하십시오.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import build_lfw_manifest, write_lfw_manifest_bundle

print(f'project_root: {PROJECT_ROOT}')
print(f'python: {sys.executable}')
print(f'pandas: {pd.__version__}')


project_root: D:\ronbun
python: c:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
pandas: 3.0.3


## 1. 경로와 분할 조건 고정

기본 `MIN_GALLERY_IMAGES=6`은 enrollment count 5를 사용해도 identity마다 등록 probe가 최소 1장 남도록 하기 위한 조건입니다. 같은 논문 결과를 재현하려면 아래 값과 생성된 `summary.json`을 함께 보존하십시오.


In [6]:
WRITE_OUTPUTS = True  # 검증 완료 후에만 True
OVERWRITE = False      # 기존 결과를 의도적으로 교체할 때만 True

SEED = 42
DEVELOPMENT_FRACTION = 0.60
CALIBRATION_FRACTION = 0.20
GALLERY_SIZE = 50
MIN_GALLERY_IMAGES = 6
UNKNOWN_UNKNOWN_FRACTION = 0.50

LFW_ROOT = PROJECT_ROOT / 'data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled'
OUTPUT_DIR = PROJECT_ROOT / 'data/interim/lfw'

display(pd.Series({
    'WRITE_OUTPUTS': WRITE_OUTPUTS,
    'OVERWRITE': OVERWRITE,
    'LFW_ROOT': str(LFW_ROOT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'SEED': SEED,
    'GALLERY_SIZE': GALLERY_SIZE,
    'MIN_GALLERY_IMAGES': MIN_GALLERY_IMAGES,
}, name='value').to_frame())


,value
WRITE_OUTPUTS,True
OVERWRITE,False
LFW_ROOT,D:\ronbun\data\raw\LFW\lfw-deepfunneled\lfw-de...
OUTPUT_DIR,D:\ronbun\data\interim\lfw
SEED,42
GALLERY_SIZE,50
MIN_GALLERY_IMAGES,6


## 2. 전체 데이터 검사와 매니페스트 구성

이 셀은 저장 여부와 무관하게 모든 identity 폴더와 이미지 경로를 읽습니다. 같은 identity가 여러 split에 들어가거나, 이미지 ID·경로가 중복되거나, gallery 조건을 만족하는 test identity가 부족하면 즉시 중단합니다.


In [7]:
bundle = build_lfw_manifest(
    LFW_ROOT,
    PROJECT_ROOT,
    seed=SEED,
    development_fraction=DEVELOPMENT_FRACTION,
    calibration_fraction=CALIBRATION_FRACTION,
    gallery_size=GALLERY_SIZE,
    min_gallery_images=MIN_GALLERY_IMAGES,
    unknown_unknown_fraction=UNKNOWN_UNKNOWN_FRACTION,
)

display(pd.Series(bundle.summary, name='value').to_frame())
display(
    bundle.manifest.groupby('split').agg(
        images=('image_id', 'size'),
        identities=('identity_id', 'nunique'),
    )
)
display(bundle.manifest.head(10))
print('LFW 매니페스트 검증 통과')


,value
dataset,lfw-deepfunneled
seed,42
image_count,13233
identity_count,5749
development_identity_count,3449
calibration_identity_count,1149
test_identity_count,1151
gallery_identity_count,50
known_unknown_identity_count,551
unknown_unknown_identity_count,550


,images,identities
split,,
calibration,3050,1149
development,7762,3449
test,2421,1151


,image_id,identity_id,split,image_path
0,lfw:Aaron_Pena:Aaron_Pena_0001,lfw:Aaron_Pena,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
1,lfw:Abdullatif_Sener:Abdullatif_Sener_0001,lfw:Abdullatif_Sener,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
2,lfw:Abdullatif_Sener:Abdullatif_Sener_0002,lfw:Abdullatif_Sener,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
3,lfw:Adam_Kennedy:Adam_Kennedy_0001,lfw:Adam_Kennedy,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
4,lfw:Adoor_Gopalakarishnan:Adoor_Gopalakarishna...,lfw:Adoor_Gopalakarishnan,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
5,lfw:Adriana_Perez_Navarro:Adriana_Perez_Navarr...,lfw:Adriana_Perez_Navarro,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
6,lfw:Ahmed_Ibrahim_Bilal:Ahmed_Ibrahim_Bilal_0001,lfw:Ahmed_Ibrahim_Bilal,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
7,lfw:Ahmed_Lopez:Ahmed_Lopez_0001,lfw:Ahmed_Lopez,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
8,lfw:Ai_Sugiyama:Ai_Sugiyama_0001,lfw:Ai_Sugiyama,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
9,lfw:Ai_Sugiyama:Ai_Sugiyama_0002,lfw:Ai_Sugiyama,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...


LFW 매니페스트 검증 통과


## 3. 결과 저장

기본값에서는 저장 예정 경로만 보여 줍니다. 실제 파일이 필요하면 위 설정 셀의 `WRITE_OUTPUTS=True`로 변경하고 노트북 전체를 다시 실행하십시오. 기존 파일이 있으면 `OVERWRITE=False`가 덮어쓰기를 막습니다.


In [8]:
planned_paths = {
    name: OUTPUT_DIR / name
    for name in (
        'face_manifest.csv',
        'gallery_identities.txt',
        'unknown_unknown_identities.txt',
        'summary.json',
    )
}

if WRITE_OUTPUTS:
    written_paths = write_lfw_manifest_bundle(
        bundle, OUTPUT_DIR, overwrite=OVERWRITE
    )
    print('저장 완료')
else:
    written_paths = planned_paths
    print('WRITE_OUTPUTS=False: 검증만 완료했으며 파일은 저장하지 않았습니다.')

display(pd.DataFrame(
    [{'file': name, 'path': str(path), 'exists': path.is_file()}
     for name, path in written_paths.items()]
))


저장 완료


,file,path,exists
0,face_manifest.csv,D:\ronbun\data\interim\lfw\face_manifest.csv,True
1,gallery_identities.txt,D:\ronbun\data\interim\lfw\gallery_identities.txt,True
2,unknown_unknown_identities.txt,D:\ronbun\data\interim\lfw\unknown_unknown_ide...,True
3,summary.json,D:\ronbun\data\interim\lfw\summary.json,True


## 다음 단계

저장 후 `configs/experiments/face_search.yaml`의 경로를 다음처럼 맞춘 뒤 `00_protocol_and_run_freeze.ipynb`를 실행합니다.

```yaml
dataset:
  manifest_path: data/interim/lfw/face_manifest.csv
protocol:
  gallery_identities_path: data/interim/lfw/gallery_identities.txt
  unknown_unknown_identities_path: data/interim/lfw/unknown_unknown_identities.txt
```

이 노트북은 설정 파일을 자동 변경하지 않습니다. LFW와 SurvFace 설정이 섞이지 않도록 실험별 config를 복사해 사용하는 편이 안전합니다.
